Simulated Backtest

In [2]:
import yfinance as yf
import pandas as pd

BIG_SMA = 100
SMALL_SMA = 50
STOP_LOSS_PERCENTAGE = 0.02
RR_RATIO = 2
ACCOUNT_SIZE = 10000
PERCENTAGE_INVESTED = 1.0

df = yf.download("SPY", period="2y", interval="4h", multi_level_index=False)

[*********************100%***********************]  1 of 1 completed


In [7]:
df["SMA100"] = df["Close"].rolling(BIG_SMA).mean()
df["SMA50"] = df["Close"].rolling(SMALL_SMA).mean()

df["Change"] = df["Close"] - df["Close"].shift(1)
df["Increase"] = df["Change"].clip(lower=0)
df["Decrease"] = (df["Change"].clip(upper=0)).abs()

df["RSI"] = 100 - (100 / (1 + df["Increase"].rolling(14).mean() / df["Decrease"].rolling(14).mean()))
df.dropna(inplace=True)

In [14]:
in_position = False
trades = []
equity = ACCOUNT_SIZE

for row in df.itertuples():
    if row.RSI < 30:
        if not in_position:
            print(f"Buy at {row.Close}")
            in_position = True
            trades.append({"Entry Date": pd.to_datetime(row.Index), "Exit Date": None, "Entry Price": row.Close, "Stop Loss": row.Close * (1 - STOP_LOSS_PERCENTAGE), "Take Profit": row.Close * (1 + STOP_LOSS_PERCENTAGE * RR_RATIO)})
    elif row.RSI > 70:
        if in_position:
            print(f"Sell at {row.Close}")
            in_position = False
            trades[-1]["Exit Price"] = row.Close
            trades[-1]["Exit Date"] = pd.to_datetime(row.Index)
            equity += (trades[-1]["Exit Price"] - trades[-1]["Entry Price"]) * equity * PERCENTAGE_INVESTED / trades[-1]["Entry Price"]

    if in_position:
        if row.Close <= trades[-1]["Stop Loss"]:
            print(f"Stop Loss hit at {row.Close}")
            in_position = False
            trades[-1]["Exit Price"] = row.Close
            trades[-1]["Exit Date"] = pd.to_datetime(row.Index)
            equity += (trades[-1]["Exit Price"] - trades[-1]["Entry Price"]) * equity * PERCENTAGE_INVESTED / trades[-1]["Entry Price"]
        elif row.Close >= trades[-1]["Take Profit"]:
            print(f"Take Profit hit at {row.Close}")
            in_position = False
            trades[-1]["Exit Price"] = row.Close
            trades[-1]["Exit Date"] = pd.to_datetime(row.Index)
            equity += (trades[-1]["Exit Price"] - trades[-1]["Entry Price"]) * equity * PERCENTAGE_INVESTED / trades[-1]["Entry Price"]

if trades and"Exit Price" not in trades[-1]:
    trades[-1]["Exit Price"] = df.iloc[-1]["Close"]
    trades[-1]["Exit Date"] = pd.to_datetime(df.index[-1])
    equity += (trades[-1]["Exit Price"] - trades[-1]["Entry Price"]) * equity * PERCENTAGE_INVESTED / trades[-1]["Entry Price"]

trades_df = pd.DataFrame(trades)
trades_df["PnL"] = trades_df["Exit Price"] - trades_df["Entry Price"]
trades_df["PnL %"] = trades_df["PnL"] / trades_df["Entry Price"] * 100

print(f"Total PnL: {trades_df['PnL'].sum():.2f}")
print(f"Total PnL %: {((1 + (trades_df['PnL %'] / 100)).cumprod().iloc[-1] - 1) * 100:.2f}%")
print(f"Total Equity: {equity:.2f}")

Buy at 594.2100219726562
Stop Loss hit at 577.5399780273438
Buy at 576.8800048828125
Stop Loss hit at 561.8400268554688
Buy at 552.2000122070312
Sell at 573.0499877929688
Buy at 542.260009765625
Stop Loss hit at 511.1600036621094
Buy at 505.510009765625
Take Profit hit at 542.8400268554688
Buy at 620.5700073242188
Sell at 641.5399780273438
Buy at 634.4349975585938
Sell at 657.1199951171875
Buy at 653.1099853515625
Sell at 671.239990234375
Buy at 670.9600219726562
Stop Loss hit at 652.530029296875
Buy at 658.3599853515625
Sell at 682.47998046875
Buy at 657.1199951171875
Stop Loss hit at 636.6199951171875
Buy at 634.0800170898438
Sell at 675.010009765625
Total PnL: 83.27
Total PnL %: 13.78%
Total Equity: 11378.13
